# Construcción del Modelo Predictivo de Insatisfacción


## 1. Carga de datos y librerías

Importamos las librerías necesarias y cargamos los datos de quejas de clientes.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Cargar datos
ruta_datos = r"../data/BD_Quejas_Analitica.xlsx"
df = pd.read_excel(ruta_datos)
df["Mes apertura del caso"] = df["Mes apertura del caso"].astype(str)
df.head()

,Mes apertura del caso,Descripción,Tipo,Nombre del cliente,Canal de comunicación,Categoria_Predicha,Probabilidad
0,202501,USUARIA SOLICITA SABER EL ESTADO DE UNA INCAPA...,QUEJA,JORGE MEJIA,LINEA DE ATENCIÓN,NaN,NaN
1,202501,BUEN DÍA CANDIALMENTE SOLICITAMOS INFORMACIÓN ...,QUEJA,FERLEY GANAN,SEGUROSSURA.COM.CO,NaN,NaN
2,202501,"BUENAS DÍAS , QUIERO UN CERTIFICADO DE INCAPAC...",QUEJA,FABIAN LOPEZ,SEGUROSSURA.COM.CO,NaN,NaN
3,202501,USUARIO SOLICITA SABER EL ESTADO DE SU INCAPAC...,QUEJA,SAMAN INVERSIONES S.A.S.,LINEA DE ATENCIÓN,NaN,NaN
4,202501,POR MEDIO DE LA PRESENTE SOLICITO INFORMACION ...,QUEJA,JOSE CHAVERRA,SEGUROSSURA.COM.CO,NaN,NaN


## 2. Feature Engineering


In [2]:
# Función para clasificar quejas (de src/metricas.py)
def classify_queja(text):
    text = str(text).upper()
    if ("PAGO" in text or "COBRO" in text or "MORA" in text or "SALARIO" in text or "PLATA" in text) and "INCAPACIDAD" in text:
        return "DEMORA_PAGO_INCAPACIDAD"
    if "INCAPACIDAD" in text and any(w in text for w in ["ESTADO", "INFORMACIÓN", "INFORMACION", "CONSULTA", "AVERIGUAR", "SABER"]):
        return "CONSULTA_ESTADO_INCAPACIDAD"
    if "INCAPACIDAD" in text and "RADICADO" in text:
        return "SEGUIMIENTO_RADICADO"
    if "CERTIFICADO" in text:
        return "SOLICITUD_CERTIFICADO"
    if "ACCIDENTE" in text and "TRABAJO" in text:
        return "ACCIDENTE_TRABAJO"
    if "INCAPACIDAD" in text:
        return "GESTION_INCAPACIDAD"
    return "OTRA_SOLICITUD"

# Construcción de features a nivel cliente
df["Categoria"] = df["Descripción"].apply(classify_queja)
feat = df.groupby("Nombre del cliente").agg(
    total_quejas=("Descripción", "count"),
    canales_distintos=("Canal de comunicación", "nunique"),
    meses_distintos=("Mes apertura del caso", "nunique"),
    tiene_pago=("Categoria", lambda x: int("DEMORA_PAGO_INCAPACIDAD" in x.values)),
    tiene_consulta=("Categoria", lambda x: int("CONSULTA_ESTADO_INCAPACIDAD" in x.values)),
    tiene_accidente=("Categoria", lambda x: int("ACCIDENTE_TRABAJO" in x.values)),
    canal_principal=("Canal de comunicación", lambda x: x.value_counts().index[0]),
).reset_index()

# Codificar canal principal
le = LabelEncoder()
feat["canal_enc"] = le.fit_transform(feat["canal_principal"])

# Variable objetivo: alto riesgo = 3+ quejas
UMBRAL = 3
feat["alto_riesgo"] = (feat["total_quejas"] >= UMBRAL).astype(int)
feat.head()

,Nombre del cliente,total_quejas,canales_distintos,meses_distintos,tiene_pago,tiene_consulta,tiene_accidente,canal_principal,canal_enc,alto_riesgo
0,ABEL ARBOLEDA,1,1,1,1,0,0,LINEA DE ATENCIÓN,1,0
1,ABEL Y SOFIA SAS,1,1,1,0,0,0,SEGUROSSURA.COM.CO,3,0
2,ABORIIGEN S.A.S,1,1,1,1,0,0,LINEA DE ATENCIÓN,1,0
3,ACABADOS SUAREZ J.S. S.A.S.,2,1,2,1,0,0,SEGUROSSURA.COM.CO,3,0
4,ACCION Y MISION CYG SAS,1,1,1,1,0,0,SEGUROSSURA.COM.CO,3,0


## 3. Entrenamiento del modelo

Entrenamos un modelo Random Forest para predecir clientes con alto riesgo de insatisfacción.

In [3]:
feature_cols = [
    "total_quejas", "canales_distintos", "meses_distintos",
    "tiene_pago", "tiene_consulta", "tiene_accidente", "canal_enc"
]
X = feat[feature_cols]
y = feat["alto_riesgo"]

modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

scores = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc")
print(f"AUC-ROC (CV-5): {scores.mean():.3f} ± {scores.std():.3f}")

modelo.fit(X, y)

# Importancia de variables
importancias = pd.Series(modelo.feature_importances_, index=feature_cols)
importancias.sort_values(ascending=False)

AUC-ROC (CV-5): 1.000 ± 0.000


total_quejas         0.620469
meses_distintos      0.252590
canales_distintos    0.076650
tiene_pago           0.027002
canal_enc            0.012762
tiene_consulta       0.009727
tiene_accidente      0.000799
dtype: float64

## 4. Predicción y ranking de clientes

Calculamos la probabilidad de insatisfacción para cada cliente y generamos un ranking.

In [4]:
# Predicción de probabilidad de insatisfacción
df_pred = feat.copy()
df_pred["prob_insatisfaccion"] = modelo.predict_proba(X)[:, 1]
df_pred["nivel_riesgo"] = pd.cut(
    df_pred["prob_insatisfaccion"],
    bins=[0, 0.3, 0.6, 1.0],
    labels=["BAJO", "MEDIO", "ALTO"]
)

ranking = df_pred[["Nombre del cliente", "total_quejas", "prob_insatisfaccion", "nivel_riesgo"]] \
    .sort_values("prob_insatisfaccion", ascending=False)

ranking.head(15)

,Nombre del cliente,total_quejas,prob_insatisfaccion,nivel_riesgo
638,JAIDER TAPIA,4,1.000000,ALTO
751,JOSE VEGA,4,0.999387,ALTO
217,CESAR CORTES,5,0.999387,ALTO
659,JERUSCELKYS SANCHEZ,6,0.998907,ALTO
446,EVER IMBACHI,6,0.998575,ALTO
386,DUBERNEY RAMIREZ,6,0.998575,ALTO
298,CREZCAMOS S.A. COMPAÑIA DE FINANCIAMIENTO,4,0.998575,ALTO
882,MANUEL GIRALDO,3,0.998575,ALTO
675,JHON MONTANO,5,0.998195,ALTO
1205,SONIA DOMINGUEZ,5,0.998195,ALTO


## 5. Exportar resultados

Exportamos el ranking de clientes a un archivo Excel para su análisis posterior.

In [5]:
# Exportar ranking a Excel
ranking.to_excel(r"../outputs/ranking_riesgo_clientes.xlsx", index=False)
print("Ranking exportado a ../outputs/ranking_riesgo_clientes.xlsx")

Ranking exportado a ../outputs/ranking_riesgo_clientes.xlsx
